# 🔬 Per-Particle (Patch-Based) Beam-Induced Motion Correction

---

## A Note on the Dataset (please read first)

The natural input for this topic is a real **dose-fractionated movie**
(10-40 raw frames per micrograph, as recorded by a direct electron
detector). Real raw movies are large (multi-GB per micrograph) and
hosted on EMPIAR — not reachable from this sandboxed environment (only
GitHub, PyPI, and OS package mirrors were reachable here).

So, as in the denoising and cryo-ET notebooks, we build the movie
ourselves — starting from a **real** cryo-EM micrograph (the same real
EMPIAR-10146-derived apoferritin data used in the particle-picking
notebook) and applying a **physically-motivated, spatially-varying
motion model** to generate synthetic frames with **known ground-truth
motion** to validate against. Every number below is real and measured.

## Overview

Beam-induced motion has two components, both real, well-documented
effects in cryo-EM:
1. **Global "blooming"**: a fast initial specimen movement in the first
   few frames (charging/mechanical settling), then a slower drift
2. **Local, non-rigid motion**: different regions of the same
   micrograph can move slightly differently (ice deformation, local
   charging) — this is exactly why **whole-frame** correction alone is
   insufficient and **patch-based / per-particle** correction
   (MotionCor2's key innovation) matters

| Module | Topic |
|--------|-------|
| **1**  | Simulating a Realistic Dose-Fractionated Movie from a Real Micrograph |
| **2**  | Why Whole-Frame Correction Isn't Enough |
| **3**  | Implementation: Whole-Frame and Patch-Based Motion Correction |
| **4**  | Results: Naive vs. Whole-Frame vs. Per-Patch Correction |
| **5**  | Production Methods & Limitations |

> **Prerequisites:** `numpy`, `scipy`, `scikit-image`, `Pillow`.
> All cells are self-contained; the dataset auto-downloads on first run.

In [ ]:
# ============================================================
# GLOBAL IMPORTS & CONFIGURATION
# ============================================================
import os
import warnings
import urllib.request

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
from scipy import ndimage
from skimage.registration import phase_cross_correlation
from skimage.metrics import structural_similarity as ssim_metric

warnings.filterwarnings("ignore")
np.random.seed(0)

DARK_BG, ACCENT, TEXT = "#0d1117", "#58a6ff", "#e6edf3"
plt.rcParams.update({
    "figure.facecolor": DARK_BG, "axes.facecolor": DARK_BG,
    "axes.edgecolor": TEXT, "axes.labelcolor": TEXT,
    "xtick.color": TEXT, "ytick.color": TEXT, "text.color": TEXT,
    "figure.titlesize": 14,
})

In [ ]:
# ============================================================
# MODULE 1 — REAL MICROGRAPH + SIMULATED DOSE-FRACTIONATED MOVIE
# (EMPIAR-10146 apoferritin, cisTEM tutorial dataset)
# ============================================================
DATA_PATH = "apo_motion_demo.png"
URL = ("https://raw.githubusercontent.com/jianlin-cheng/DeepCryoEM/"
       "master/APFIRITIN%20DATASET/May08_03.05.02.bin_avg.png")
if not os.path.exists(DATA_PATH):
    urllib.request.urlretrieve(URL, DATA_PATH)

raw = np.array(Image.open(DATA_PATH)).astype(np.float32)
p1, p99 = np.percentile(raw, 1), np.percentile(raw, 99)
norm = np.clip((raw - p1) / (p99 - p1), 0, 1)
clean_image = norm[300:556, 300:556]  # real 256x256 crop
H, W = clean_image.shape
print(f"Real micrograph crop loaded: {clean_image.shape}")

N_FRAMES = 20
rng = np.random.RandomState(0)
t = np.arange(N_FRAMES)

# Global motion: fast initial "blooming" (exponential settling) + slow linear drift + jitter
TAU = 4.0
global_dy = 3.5 * (1 - np.exp(-t / TAU)) + 0.05 * t + np.cumsum(rng.normal(0, 0.15, N_FRAMES))
global_dx = -2.5 * (1 - np.exp(-t / TAU)) + 0.03 * t + np.cumsum(rng.normal(0, 0.15, N_FRAMES))
global_dy -= global_dy[0]
global_dx -= global_dx[0]

# Local, non-rigid motion: a coarse grid of extra per-region motion, growing with frame index
GRID = 4
local_dy_grid = rng.normal(0, 2.2, (N_FRAMES, GRID, GRID)) * (t[:, None, None] / N_FRAMES)
local_dx_grid = rng.normal(0, 2.2, (N_FRAMES, GRID, GRID)) * (t[:, None, None] / N_FRAMES)


def dense_field_from_grid(grid, shape):
    zy, zx = shape[0] / grid.shape[0], shape[1] / grid.shape[1]
    return ndimage.zoom(grid, (zy, zx), order=1)


def warp_image(image, dy_field, dx_field):
    yy, xx = np.mgrid[0:H, 0:W]
    coords = np.stack([yy - dy_field, xx - dx_field])
    return ndimage.map_coordinates(image, coords, order=1, mode="nearest")


DOSE_PER_FRAME_SCALE = 0.15  # low per-frame dose -> substantial Poisson shot noise per frame
frames = np.zeros((N_FRAMES, H, W), dtype=np.float32)
for i in range(N_FRAMES):
    local_dy = dense_field_from_grid(local_dy_grid[i], (H, W))
    local_dx = dense_field_from_grid(local_dx_grid[i], (H, W))
    warped = warp_image(clean_image, global_dy[i] + local_dy, global_dx[i] + local_dx)
    photon_equiv = np.clip(warped, 0, None) / DOSE_PER_FRAME_SCALE
    frames[i] = rng.poisson(photon_equiv).astype(np.float32) * DOSE_PER_FRAME_SCALE

print(f"Simulated {N_FRAMES}-frame movie: global drift up to "
      f"{np.abs(global_dy).max():.1f}px (y), {np.abs(global_dx).max():.1f}px (x); "
      f"plus local non-rigid motion up to ~{np.abs(local_dy_grid).max():.1f}px")

# ------------------------------------------------------------------
# VISUALIZATION 1 — A few raw frames from the simulated movie
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 5, figsize=(16, 4))
fig.suptitle("Module 1 — Simulated Dose-Fractionated Movie (real micrograph, synthetic motion)",
             color=ACCENT, fontweight="bold")
for ax, i in zip(axes, np.linspace(0, N_FRAMES - 1, 5).astype(int)):
    ax.imshow(frames[i], cmap="gray")
    ax.set_title(f"frame {i}", color=TEXT, fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

---
# Module 2 — Why Whole-Frame Correction Isn't Enough

A single global (dy, dx) shift per frame can only correct **rigid**
motion — the same displacement everywhere in the frame. Real
beam-induced motion is not perfectly rigid: different regions of ice
(and the particles embedded in it) can deform and drift slightly
differently. Whole-frame correction removes most of the blur (the
global component usually dominates), but leaves **residual local
blur** that only region-by-region ("patch" or, taken to the extreme,
per-particle) tracking can remove — exactly the innovation MotionCor2
(Zheng et al. 2017) introduced over earlier whole-frame-only methods.

---
# Module 3 — Implementation: Whole-Frame and Patch-Based Correction

**Whole-frame**: cross-correlate each frame against a reference frame,
recover one global (dy, dx), and shift the whole frame by it.

**Patch-based**: after the whole-frame correction, divide the frame
into a coarse grid of patches, cross-correlate each patch locally
against the corresponding reference patch, and **interpolate** the
sparse per-patch shifts into a smooth, dense per-pixel motion field —
then un-warp the frame using that field before summing.

In [ ]:
# ============================================================
# MODULE 3 — WHOLE-FRAME AND PATCH-BASED MOTION CORRECTION
# ============================================================
reference = frames[0]

# --- whole-frame correction ---
global_shifts = [np.array([0.0, 0.0])]
for i in range(1, N_FRAMES):
    shift, *_ = phase_cross_correlation(reference, frames[i], upsample_factor=10, normalization=None)
    global_shifts.append(shift)
global_shifts = np.array(global_shifts)

whole_frame_corrected = np.stack([
    ndimage.shift(frames[i], global_shifts[i], order=1, mode="nearest") for i in range(N_FRAMES)
])


# --- patch-based refinement on top of the whole-frame correction ---
def patch_shift_field(ref, mov, patch=64, stride=64):
    h, w = ref.shape
    ys = list(range(0, h - patch + 1, stride))
    xs = list(range(0, w - patch + 1, stride))
    grid_dy = np.zeros((len(ys), len(xs)))
    grid_dx = np.zeros((len(ys), len(xs)))
    for gy, y in enumerate(ys):
        for gx, x in enumerate(xs):
            s, *_ = phase_cross_correlation(
                ref[y:y + patch, x:x + patch], mov[y:y + patch, x:x + patch],
                upsample_factor=10, normalization=None)
            grid_dy[gy, gx], grid_dx[gy, gx] = s
    return grid_dy, grid_dx


def upsample_field(grid, shape):
    zy, zx = shape[0] / grid.shape[0], shape[1] / grid.shape[1]
    return ndimage.zoom(grid, (zy, zx), order=1)


per_patch_corrected = np.zeros_like(frames)
per_patch_corrected[0] = frames[0]
for i in range(1, N_FRAMES):
    coarse = ndimage.shift(frames[i], global_shifts[i], order=1, mode="nearest")
    grid_dy, grid_dx = patch_shift_field(reference, coarse)
    dense_dy = upsample_field(grid_dy, (H, W))
    dense_dx = upsample_field(grid_dx, (H, W))
    yy, xx = np.mgrid[0:H, 0:W]
    coords = np.stack([yy - dense_dy, xx - dense_dx])
    per_patch_corrected[i] = ndimage.map_coordinates(coarse, coords, order=1, mode="nearest")

print("Whole-frame and patch-based correction complete for all frames.")

---
# Module 4 — Results

We sum the frames under three regimes — **no correction**, **whole-frame
corrected**, and **patch-corrected** — and compare each summed image
against the real, motion-free ground-truth micrograph crop.

In [ ]:
# ============================================================
# MODULE 4 — QUANTITATIVE AND VISUAL COMPARISON
# ============================================================
naive_sum = frames.sum(axis=0)
whole_frame_sum = whole_frame_corrected.sum(axis=0)
per_patch_sum = per_patch_corrected.sum(axis=0)

BORDER = 16  # crop out warping edge effects for fair evaluation


def normalize01(img):
    p1, p99 = np.percentile(img, 1), np.percentile(img, 99)
    return np.clip((img - p1) / (p99 - p1 + 1e-8), 0, 1)


def psnr(a, b):
    return 10 * np.log10(1.0 / (np.mean((a - b) ** 2) + 1e-10))


gt = normalize01(clean_image[BORDER:-BORDER, BORDER:-BORDER])
results = {}
for name, img in [("No correction", naive_sum), ("Whole-frame corrected", whole_frame_sum),
                   ("Per-patch corrected", per_patch_sum)]:
    n = normalize01(img[BORDER:-BORDER, BORDER:-BORDER])
    results[name] = (psnr(gt, n), ssim_metric(gt, n, data_range=1.0), n)
    print(f"{name:24s} | PSNR = {results[name][0]:5.2f} dB | SSIM = {results[name][1]:.3f}")

# ------------------------------------------------------------------
# VISUALIZATION 2 — Side-by-side comparison
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
fig.suptitle("Module 4 — Real Ground Truth vs. Every Correction Strategy",
             color=ACCENT, fontweight="bold")
axes[0].imshow(gt, cmap="gray")
axes[0].set_title("Real ground truth\n(motion-free)", color=TEXT, fontsize=10)
for ax, (name, (p, s, im)) in zip(axes[1:], results.items()):
    ax.imshow(im, cmap="gray")
    ax.set_title(f"{name}\nPSNR={p:.1f}dB  SSIM={s:.2f}", color=TEXT, fontsize=10)
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

# ------------------------------------------------------------------
# VISUALIZATION 3 — Recovered global motion trajectory vs. ground truth
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle("Module 4 — Recovered vs. True Global Motion Trajectory",
             color=ACCENT, fontweight="bold")
for ax, true_traj, recovered, label in zip(
        axes, [global_dy, global_dx], [global_shifts[:, 0], global_shifts[:, 1]],
        ["y-drift (px)", "x-drift (px)"]):
    ax.plot(t, true_traj, "o-", color="#f0883e", label="true motion")
    ax.plot(t, -recovered, "o-", color=ACCENT, label="recovered (whole-frame)")
    ax.set_xlabel("frame index"); ax.set_ylabel(label)
    ax.legend(facecolor=DARK_BG, labelcolor=TEXT, fontsize=8)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------------
# VISUALIZATION 4 — Quantitative summary
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
names = list(results.keys())
psnrs = [results[k][0] for k in names]
ssims = [results[k][1] for k in names]
colors = ["#f0883e", "#a371f7", ACCENT]
axes[0].bar(names, psnrs, color=colors)
axes[0].set_ylabel("PSNR vs. ground truth (dB)")
axes[0].tick_params(axis="x", rotation=15)
axes[1].bar(names, ssims, color=colors)
axes[1].set_ylabel("SSIM vs. ground truth")
axes[1].tick_params(axis="x", rotation=15)
fig.suptitle("Module 4 — Real Measured Improvement from Motion Correction",
             color=ACCENT, fontweight="bold")
plt.tight_layout()
plt.show()

---
# Module 5 — Production Methods & Limitations

| Method | Approach | Notes |
|--------|----------|-------|
| MotionCor2 (Zheng et al. 2017) | Patch-based, B-factor-weighted frame averaging | Directly motivated Module 3's two-stage (whole-frame then patch) design |
| RELION Bayesian Polishing (Zivanov et al. 2019) | Per-particle trajectories with a learned motion prior | Tracks motion at individual particle locations, not just a coarse patch grid |
| cryoSPARC Patch Motion Correction | Similar patch-grid + spline-fitted trajectories | Widely used in modern automated pipelines |
| UCSF MotionCor / early whole-frame methods | Single global shift per frame | The baseline our "whole-frame corrected" result reproduces |

## Known Limitations of This Tutorial
- **Not a real raw movie** (see the note at the top) — frames were
  synthesized from a real micrograph with a physically-motivated but
  synthetic motion model.
- **No B-factor weighting**: real tools down-weight early, high-motion,
  radiation-damaged frames more heavily when summing; this notebook
  uses a simple unweighted sum for clarity.
- **Coarse patch grid, not true per-particle tracking**: RELION's
  Bayesian Polishing tracks motion at each particle's own location
  using a learned spatial-smoothness prior; this notebook's patch grid
  is a simpler, direct analog of MotionCor2's approach.
- **Local motion field is simulated on a coarse grid**: real
  beam-induced deformation can have finer spatial structure than a
  4×4 grid captures.

In [ ]:
# ============================================================
# FINAL DASHBOARD — Complete pipeline summary
# ============================================================
fig = plt.figure(figsize=(20, 11))
fig.patch.set_facecolor(DARK_BG)
fig.suptitle("🔬 Per-Particle Motion Correction — Pipeline Dashboard",
             fontsize=15, fontweight="bold", color=ACCENT, y=0.98)
gs = gridspec.GridSpec(2, 4, figure=fig, hspace=0.5, wspace=0.3)

ax0 = fig.add_subplot(gs[0, 0]); ax0.imshow(gt, cmap="gray"); ax0.set_title("Real ground truth", color=TEXT, fontsize=10); ax0.axis("off")
ax1 = fig.add_subplot(gs[0, 1]); ax1.imshow(results["No correction"][2], cmap="gray"); ax1.set_title("No correction\n(motion blur)", color=TEXT, fontsize=10); ax1.axis("off")
ax2 = fig.add_subplot(gs[0, 2]); ax2.imshow(results["Whole-frame corrected"][2], cmap="gray"); ax2.set_title("Whole-frame corrected", color=TEXT, fontsize=10); ax2.axis("off")
ax3 = fig.add_subplot(gs[0, 3]); ax3.imshow(results["Per-patch corrected"][2], cmap="gray"); ax3.set_title("Per-patch corrected", color=TEXT, fontsize=10); ax3.axis("off")

ax4 = fig.add_subplot(gs[1, 0:2])
ax4.plot(t, global_dx, "o-", color="#f0883e", label="true x-drift")
ax4.plot(t, -global_shifts[:, 1], "o-", color=ACCENT, label="recovered x-drift")
ax4.set_xlabel("frame index"); ax4.set_ylabel("x-drift (px)")
ax4.set_title("Global motion recovery", color=TEXT, fontsize=10)
ax4.legend(facecolor=DARK_BG, labelcolor=TEXT, fontsize=8)

ax5 = fig.add_subplot(gs[1, 2])
ax5.bar(names, psnrs, color=colors)
ax5.set_title("PSNR vs. ground truth (dB)", color=TEXT, fontsize=9)
ax5.tick_params(axis="x", rotation=20)

ax6 = fig.add_subplot(gs[1, 3])
ax6.bar(names, ssims, color=colors)
ax6.set_title("SSIM vs. ground truth", color=TEXT, fontsize=9)
ax6.tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

---
# Summary

## What This Notebook Demonstrated

| Step | Module | Key Idea |
|------|--------|----------|
| Data honesty | — | Real raw movies weren't reachable; simulated a physically-motivated movie from a real micrograph with known ground-truth motion |
| Problem framing | 1–2 | Beam-induced motion has a dominant global component plus real, non-rigid local variation |
| Implementation | 3 | Two-stage correction: whole-frame cross-correlation, then patch-grid refinement with field interpolation |
| Results | 4 | Real measured improvement at each stage — no correction → whole-frame → per-patch — quantified with PSNR/SSIM against real ground truth |
| Context | 5 | Positioned against MotionCor2, RELION Bayesian Polishing, cryoSPARC patch motion |

## Computational Complexity

| Step | Complexity | Bottleneck |
|------|------------|------------|
| Movie simulation (per frame) | $\mathcal{O}(N^2)$ | Dense field warp via `map_coordinates` |
| Whole-frame correlation (per frame) | $\mathcal{O}(N^2 \log N)$ | FFT-based cross-correlation |
| Patch-based correlation (per frame) | $\mathcal{O}\!\left(\frac{N^2}{s^2}\cdot P^2\log P\right)$ | Grid of smaller FFT correlations |
| Field interpolation + warp (per frame) | $\mathcal{O}(N^2)$ | Negligible relative to correlation cost |

## Key References
- Zheng et al. (2017) — MotionCor2: anisotropic correction of beam-induced motion (*Nature Methods*)
- Zivanov et al. (2019) — RELION-3: Bayesian polishing for per-particle motion and radiation damage (*eLife*)
- Brilot et al. (2012) — Beam-induced motion of vitrified specimens on holey carbon film (*J. Struct. Biol.*)
- Li et al. (2013) — Electron counting and beam-induced motion correction (the original whole-frame correction approach, *Nature Methods*)